In [1]:
import instructor
from enum import Enum
from openai import AzureOpenAI
from functools import partial
# from Entity_response_model import Demographic, Refractory_Relapse, Blast, Diagnosis, test
from Entity_IC_test import Test,Demographcics,Inhibitors
from Entity_IC import Entity as Entity_IC
from Entity_EC import Entity as Entity_EC
import pandas as pd
from datetime import datetime
from typing import TypeVar, Type, Optional, List
from pydantic import BaseModel, Field
import os
import re
from itertools import islice


In [2]:
# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = 'YOUR-API-KEY'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    max_retries=2  # Retries in case of API issues
)


In [3]:
input_dir = '../input'
df = pd.DataFrame(columns=['File Name', 'Inclusion Criteria', 'Exclusion Criteria'])
abstract_list = [] 

for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            ### Split the notes to inclusion and exclusion criteria seperately
            inclusion_criteria = content.split('Exclusion Criteria:')[0].split('Inclusion Criteria:')[-1].strip()
            exclusion_criteria = content.split('Exclusion Criteria:')[-1].strip()
            new_row = pd.DataFrame({'File Name': [file_name], 'Inclusion Criteria': [inclusion_criteria], 'Exclusion Criteria': [exclusion_criteria]})
            df = pd.concat([df, new_row], ignore_index=True)

abstract_list = df[['File Name', 'Inclusion Criteria', 'Exclusion Criteria']].values.tolist()
print("Abstracts have been successfully read into a list.")

Abstracts have been successfully read into a list.


In [4]:
T = TypeVar("T", bound=BaseModel)

class InfectionClassifier:
    def __init__(self, response_model: Type[T], model_name: str, client):
        self.response_model = response_model
        self.model_name = model_name
        self.client = client

    def ask_ai(self, criteria:str, type:str) -> T:
        return self.client(
            response_model=self.response_model,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": "You will help extract entities from clinical protocol documents",
                },
                {
                    "role": "user",
                    "content": f"Extract information from the clinical trial eligibility document ({type} criteria).",
                },
                {
                    "role": "user",
                    "content": f"Here is the {type} Criteria section {criteria}",
                },
                {
                    "role": "user",
                    "content": f'''Please follow the following guidelines to answer the questions:
                                * Ensure you distinguish between 'CYP3A4' and 'CYP3A'. They are distinct, and an instance of 'CYP3A' should not be interpreted as 'CYP3A4'.
                                * Given content 'Refractory leukemia, defined as persistent leukemia after at least two courses of induction chemotherapy', 
                                it does not indicate occurrences of refractory disease but indicates the number of cycles of therapy required for refractory disease.
                                * Given content 'evidence of leukemia with ≥ 5% blasts in the blood', it dose not indicate blood absolute blast count but indicates blood blast percentage.
                                * Given content 'At least 24 hours must have elapsed since the completion of low-dose or non- myelosuppressive therapy', 
                                it indicates prior exposure to non-myelosuppressive anti-cancer agents.
                                * Given content 'At least 14 days must have elapsed since the completion of myelosuppressive therapy', it indicates prior exposure to myelosuppressive anti-cancer agents.
                                ''',
                },
            ],
            validation_context={"text_chunk": criteria},
        ) 

    def output(self, criteria: str, type: str):
        try:
            response = self.ask_ai(criteria, type)
            entity_dict = response.dict()

            for key, value in entity_dict.items():
                if isinstance(value, list):
                    if value and isinstance(value[0], Enum):
                        entity_dict[key] = [v.value for v in value]  # Extract the string representation
                    # else: keep the list as is
                elif isinstance(value, Enum):
                    entity_dict[key] = value.value  # For single Enum fields

            entity_value_data = {
                "Entity": list(entity_dict.keys()),
                "Value": list(entity_dict.values()),
            }
            df = pd.DataFrame(entity_value_data)
            return df
        
        except Exception as e:
            print(f"Error in output method: {str(e)}")
            # Return an empty DataFrame if an error occurs
            return pd.DataFrame(columns=["Entity", "Value"])

### Multiple categorized response model - Run3

In [6]:
# response_models = [Demographic, Refractory_Relapse, Blast, Diagnosis]  # List of response models you have
# for item in abstract_list:
#     file_name, inclusion_criteria, exclusion_criteria = item  
#     clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
#     clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
#     entity_value_list = []
#     # Iterate over each response_model
#     for response_model in response_models:
#         # Create an instance of the InfectionClassifier for each model
#         Classifier = InfectionClassifier(response_model=response_model, model_name=model, client=in_client)
        
#         # Get the inclusion and exclusion criteria as DataFrames
#         df_IC = Classifier.output(criteria=clean_text_ic, type='Inclusion')
#         df_EC = Classifier.output(criteria=clean_text_ec, type='Exclusion')
        
#         # Merge the two DataFrames on the 'Entity' column
#         df_final = pd.merge(df_IC, df_EC, on='Entity', how='outer', suffixes=('_IC', '_EC'))
        
#         # Append the resulting DataFrame to the list
#         entity_value_list.append(df_final)

#     # Concatenate all DataFrames in the list into a single DataFrame
#     df_concatenated = pd.concat(entity_value_list, ignore_index=True)
#     output_excel_path = f'../results/{file_name}_output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
#     df_concatenated.to_excel(output_excel_path, index=False)

### One response model contain all entities

In [7]:
# for item in abstract_list:
#     file_name, inclusion_criteria, exclusion_criteria = item  
#     print(file_name)
#     clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
#     clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
#     # entity_value_list = []

#     # Create an instance of the InfectionClassifier for each model
#     Classifier_IC = InfectionClassifier(response_model=Entity_IC, model_name=model, client=in_client)
#     Classifier_EC = InfectionClassifier(response_model=Entity_EC, model_name=model, client=in_client)
#     # Get the inclusion and exclusion criteria as DataFrames
#     df_IC = Classifier_IC.output(criteria=clean_text_ic, type='Inclusion')
#     df_EC = Classifier_EC.output(criteria=clean_text_ec, type='Exclusion')

#     # Merge the two DataFrames on the 'Entity' column
#     df_final = pd.merge(df_IC, df_EC, on='Entity', how='outer', suffixes=('_IC', '_EC'))

#     # Append the resulting DataFrame to the list
#     # entity_value_list.append(df_final)

#     # Concatenate all DataFrames in the list into a single DataFrame
#     # df_concatenated = pd.concat(entity_value_list, ignore_index=True)
#     output_excel_path = f'../results/{file_name}_output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
#     df_final.to_excel(output_excel_path, index=False)

### One resposne model for each entity - read ontology file first

In [5]:
# Load the uploaded Excel file to inspect its content
file_path = '../GEARBOx Ontology for LLM_In use.xlsx'
excel_data = pd.ExcelFile(file_path)

# Display the sheet names to understand the structure of the file
excel_data.sheet_names

['GEARBOx v4', 'xx-GEARBOx', 'xx-Example']

In [6]:
# Load the sheet to inspect its content
gearbox_v4_df = excel_data.parse('GEARBOx v4')
excel_data.close
# Display the first few rows of the dataframe to understand its structure
gearbox_v4_df.head()

,Parameter Name,Type,Description,Examples,MaxLength,Enum,ItemType,ItemDescription,ItemExamples,ItemEnum
0,age_min,integer,Does the eligibility criteria mention the mini...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
1,age_max,integer,Does the eligibility criteria mention the maxi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,weight_min,number,Does the eligibility criteria mention the mini...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
3,weight_max,number,Does the eligibility criteria mention the maxi...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
4,sex,string,Does the eligibility criteria mention sexes of...,NaN,15.0,"""Male"", ""Female""",NaN,NaN,NaN,NaN


In [7]:
# Function to generate class fields and enums
def generate_class_fields_and_enums(df, verb):
    fields = {}
    enum_classes = {}  # Dictionary to store generated enum classes as strings

    for _, row in df.iterrows():
        param_name = row['Parameter Name']
        param_name_exist = f"{param_name}_exist"

        # Determine the field type based on 'Type' and 'Enum' columns
        if row['Type'] in ['integer', 'number']:
            field_type = 'Optional[int]'
        elif pd.notna(row['Enum']) and 'Yes' in row['Enum']:
            field_type = 'Optional[Yes_no]'  # Assuming YesOrNo is a defined type elsewhere
        elif pd.notna(row['Enum']):
            field_type = f"Optional[List[{param_name.capitalize()}]]"
            
            # Create an enum class for this parameter
            enum_items = [item.strip().strip('"') for item in row['Enum'].split(',')]
            enum_class_str = f"class {param_name.capitalize()}(Enum):\n"
            for item in enum_items:
                if "(" in item and ")" in item:
                    enum_name = re.search(r"\((.*?)\)", item).group(1)
                    enum_name = enum_name.strip().replace(" ", "")
                    enum_name = enum_name.strip().replace("-", "").upper()
                else:
                    enum_name = item.strip().replace("-", "").replace(" ", "").upper()
                enum_class_str += f"    {enum_name} = \"{item}\"\n"
            
            enum_classes[param_name.capitalize()] = enum_class_str
            
        else:
            field_type = 'Optional[str]'
            

        # Combine description and examples if available
        description = row['Description']
        example = row['Examples']
        if pd.notna(example):
            description += f" Example: {example}"
        parts = description.split('?')
        first_yesno = parts[0] + '?'
        second_criteria = parts[1].strip().replace("{verb}", verb)
        
        fields[param_name_exist] = f"Yes_no = Field(..., description=\"{first_yesno}\")"
        fields[param_name] = f"{field_type} = Field(\n        None, \n        description=\"{second_criteria}\")"
    
    enum_items = ["Yes","No"]
    enum_class_str = f"class Yes_no(Enum):\n"
    for item in enum_items:
        enum_name = item.upper()
        enum_class_str += f"    {enum_name} = \"{item}\"\n"
    enum_classes["Yes_no"] = enum_class_str

    return fields, enum_classes


# Generate class fields from the DataFrame
class_fields_ic, enum_classes = generate_class_fields_and_enums(gearbox_v4_df, "included")
class_fields_ec, _ = generate_class_fields_and_enums(gearbox_v4_df, "excluded")

In [8]:
class_enum = ""
for enum_name, enum_definition in enum_classes.items():
    class_enum += f"{enum_definition}\n"

exec(class_enum)

In [11]:
class_enum

'class Sex(Enum):\n    MALE = "Male"\n    FEMALE = "Female"\n\nclass Bm_recent_blast_method(Enum):\n    MORPHOLOGY = "Morphology"\n    FLOWCYTOMETRY = "Flow cytometry"\n\nclass Type_transplant(Enum):\n    ALLOGENEIC = "Allogeneic"\n    AUTOLOGOUS = "Autologous"\n\nclass Expression_alk_type(Enum):\n    ALKR = "Rearrangement (ALK-R)"\n    ALKA = "Amplification (ALK-A)"\n    POINTMUTATION = "Point mutation"\n\nclass Bclc_stage(Enum):\n    STAGE0 = "Stage 0"\n    STAGEA = "Stage A"\n    STAGEB = "Stage B"\n    STAGEC = "Stage C"\n    STAGED = "Stage D"\n\nclass Yes_no(Enum):\n    YES = "Yes"\n    NO = "No"\n\n'

### One resposne model for each entity - Exp1

In [12]:
for item in abstract_list:
    file_name, inclusion_criteria, exclusion_criteria = item  
    print(file_name)
    clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
    clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')

    entity_value_list_ic = []
    # Loop through each field and process
    for field_name, field_definition in class_fields_ic.items():
        # Dynamically create the response model class
        class_basemodel_ic = f"""
class Entity(BaseModel):
    {field_name}: {field_definition}
    """
        # Compile the class definition into a Python class
        exec(class_basemodel_ic)
        response_model_ic = locals()["Entity"]

        # Initialize and use the classifier
        try:
            Classifier_IC = InfectionClassifier(response_model=response_model_ic, model_name=model, client=in_client)
            output_ic = Classifier_IC.output(criteria=clean_text_ic, type='Inclusion')
        except Exception as e:
            print(f"Error processing field '{field_name}': {e}")
        entity_value_list_ic.append(output_ic)
    df_IC = pd.concat(entity_value_list_ic, ignore_index=True)
    
    entity_value_list_ec = []
    # Loop through each field and process
    for field_name, field_definition in class_fields_ec.items():
        # Dynamically create the response model class
        class_basemodel_ec = f"""
class Entity(BaseModel):
    {field_name}: {field_definition}
    """
        # Compile the class definition into a Python class
        exec(class_basemodel_ec)
        response_model_ec = locals()["Entity"]

        # Initialize and use the classifier
        try:
            Classifier_EC = InfectionClassifier(response_model=response_model_ec, model_name=model, client=in_client)
            output_ec = Classifier_EC.output(criteria=clean_text_ec, type='Exclusion')
        except Exception as e:
            print(f"Error processing field '{field_name}': {e}")
        entity_value_list_ec.append(output_ec)
    df_EC = pd.concat(entity_value_list_ec, ignore_index=True)    
    
    # Merge the two DataFrames on the 'Entity' column
    df_final = pd.merge(df_IC, df_EC, on='Entity', how='outer', suffixes=('_IC', '_EC'))
    output_excel_path = f'../results/{file_name}_output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
    df_final.to_excel(output_excel_path, index=False)

NCT04898894.txt
NCT04915612.txt
NCT05188170.txt


### One resposne model for each entity - Exp2
To create a class_basemodel dynamically for every two pairs of field_name and field_definition from class_fields_ic

In [9]:
# Helper function to get dictionary chunks of size 2
def dict_chunk(dictionary, size=2):
    iterator = iter(dictionary.items())
    for chunk in iter(lambda: list(islice(iterator, size)), []):
        yield chunk

In [10]:
for item in abstract_list:
    file_name, inclusion_criteria, exclusion_criteria = item  
    print(file_name)
    clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
    clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')

    entity_value_list_ic = []
    for chunk in dict_chunk(class_fields_ic, size=2):
        # Dynamically create the response model class
        class_basemodel_ic = "class Entity(BaseModel):\n"
        for field_name, field_definition in chunk:
            class_basemodel_ic += f"    {field_name}: {field_definition}\n"

        # Remove any unintended leading or trailing whitespace
        class_basemodel_ic = class_basemodel_ic.strip()

        try:
            # Compile the class definition
            exec(class_basemodel_ic)
            response_model_ic = locals()["Entity"]

            # Initialize and use the classifier
            Classifier_IC = InfectionClassifier(response_model=response_model_ic, model_name=model, client=in_client)
            output_ic = Classifier_IC.output(criteria=clean_text_ic, type='Inclusion')
        except Exception as e:
            print(f"Error processing fields {[field_name for field_name, _ in chunk]}: {e}")
        entity_value_list_ic.append(output_ic)
    df_IC = pd.concat(entity_value_list_ic, ignore_index=True)

    entity_value_list_ec = []
    for chunk in dict_chunk(class_fields_ec, size=2):
        # Dynamically create the response model class
        class_basemodel_ec = "class Entity(BaseModel):\n"
        for field_name, field_definition in chunk:
            class_basemodel_ec += f"    {field_name}: {field_definition}\n"

        # Remove any unintended leading or trailing whitespace
        class_basemodel_ec = class_basemodel_ec.strip()

        try:
            # Compile the class definition
            exec(class_basemodel_ec)
            response_model_ec = locals()["Entity"]

            # Initialize and use the classifier
            Classifier_EC = InfectionClassifier(response_model=response_model_ec, model_name=model, client=in_client)
            output_ec = Classifier_EC.output(criteria=clean_text_ec, type='Inclusion')
        except Exception as e:
            print(f"Error processing fields {[field_name for field_name, _ in chunk]}: {e}")
        entity_value_list_ec.append(output_ec)
    df_EC = pd.concat(entity_value_list_ec, ignore_index=True)

    # Merge the two DataFrames on the 'Entity' column
    df_final = pd.merge(df_IC, df_EC, on='Entity', how='outer', suffixes=('_IC', '_EC'))
    output_excel_path = f'../results/{file_name}_output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
    df_final.to_excel(output_excel_path, index=False)

NCT04898894.txt
NCT04915612.txt
NCT05188170.txt
